# 🧠 Meeting Intelligence System — Kaggle Edition

**Pipeline:** Audio → Raw transcript (Whisper) → Refined transcript (Qwen2.5-7B, 4-bit) → Summary · Minutes · Decisions · Action items (Qwen2.5-3B) → **Web app (Gradio)**

### ⚙️ One-time Kaggle setup (right-hand *Notebook options* / *Session options* panel)
1. **Accelerator →** `GPU T4 x2` (best; the two models get their own GPU). `GPU P100` also works.
2. **Internet →** `On` (needed to download models and to create the public web link).
3. **Add your meeting audio:** *Add Input → Upload → New Dataset* → drop your `.m4a/.mp3/.wav` file in. The notebook auto-detects it under `/kaggle/input/`.
4. Click **Run All**, or run cell by cell. The web app is the last cell.

In [ ]:
# One install cell for everything (Kaggle: Internet must be ON)
!pip install -q -U transformers accelerate bitsandbytes sentencepiece soundfile "gradio>=5,<6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.0 MB/s eta 0:

In [ ]:
import os, sys, shutil, subprocess
import torch

IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR  = "/kaggle/working" if IS_KAGGLE else os.getcwd()
os.makedirs(WORK_DIR, exist_ok=True)

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Kaggle: Session options -> Accelerator -> GPU T4 x2, then restart the session."
    )

N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}   : {p.name}  ({p.total_memory / 1024**3:.1f} GB)")

# Device placement
#   Whisper + refiner (7B, 4-bit)  -> GPU 0
#   Documentation model (3B, fp16) -> GPU 1 when two GPUs exist, otherwise GPU 0
WHISPER_DEVICE = 0
REFINER_GPU    = 0
DOC_GPU        = 1 if N_GPUS > 1 else 0
print(f"Placement -> whisper: cuda:{WHISPER_DEVICE} | refiner: cuda:{REFINER_GPU} | documentation: cuda:{DOC_GPU}")

**PHASE 1. Raw transcript generation**

In [ ]:
# Kaggle replacement for Colab's interactive file upload:
# audio is read from an attached Kaggle Dataset under /kaggle/input/

AUDIO_PATH = "/kaggle/input/datasets/navamikhatmode/inputt/Audio Input.m4a"  # <- optionally hard-code, e.g. "/kaggle/input/my-meeting/Audio Input.m4a"
AUDIO_EXTS = (".m4a", ".mp3", ".wav", ".flac", ".ogg", ".aac", ".wma", ".mp4", ".webm", ".mkv", ".opus")

if AUDIO_PATH is None:
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files:
            if fn.lower().endswith(AUDIO_EXTS):
                candidates.append(os.path.join(root, fn))
    candidates.sort()

    if not candidates:
        raise FileNotFoundError(
            "No audio file found in /kaggle/input.\n"
            "Add one via: Add Input -> Upload -> New Dataset, or set AUDIO_PATH above."
        )

    print("Audio files found:")
    for c in candidates:
        print("  -", c)
    AUDIO_PATH = candidates[0]

print("\nUsing:", AUDIO_PATH)

In [ ]:
# ffmpeg ships with Kaggle images; install only if it is missing
if shutil.which("ffmpeg") is None:
    subprocess.run("apt-get update -qq && apt-get install -y ffmpeg -qq", shell=True, check=True)

wav_path = os.path.join(WORK_DIR, "meeting.wav")

subprocess.run([
    "ffmpeg", "-y",
    "-i", AUDIO_PATH,
    "-ar", "16000",
    "-ac", "1",
    wav_path
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Saved:", wav_path)

In [ ]:
import wave
import numpy as np
import soundfile as sf

with wave.open(wav_path, "rb") as f:
    print("Channels   :", f.getnchannels())
    print("Sample rate:", f.getframerate())
    print("Frames     :", f.getnframes())
    print("Duration   : %.1f s" % (f.getnframes() / f.getframerate()))

audio, sample_rate = sf.read(wav_path)
audio = audio.astype(np.float32)
print("Audio dtype:", audio.dtype, "| shape:", audio.shape, "| sr:", sample_rate)

Used: **Hugging Face Whisper using tranformers**

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=WHISPER_DEVICE
)

In [ ]:
result = transcriber(
    wav_path,
    return_timestamps=True,
    chunk_length_s=30,     # robust for recordings longer than 30 s
    batch_size=8
)

raw_transcript = result["text"].strip()

print("\n===== RAW TRANSCRIPT =====\n")
print(raw_transcript)

**PHASE 2. Refined transcript generation**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

REFINER_MODEL = "Qwen/Qwen2.5-7B-Instruct"

refiner_tokenizer = AutoTokenizer.from_pretrained(REFINER_MODEL)

refiner_model = AutoModelForCausalLM.from_pretrained(
    REFINER_MODEL,
    dtype=torch.float16,
    device_map={"": REFINER_GPU},
    quantization_config=quantization_config
)
print("Refiner loaded on cuda:%d" % REFINER_GPU)

In [ ]:
def refine_transcript(raw_transcript):

    prompt = f"""
You are a domain-aware meeting transcript refinement system.

Your task is to correct ONLY likely speech-to-text errors.

Focus particularly on:
- technical terminology
- scientific terminology
- engineering terms
- acronyms
- software names
- model names
- units
- numbers
- names

STRICT RULES:

1. Preserve the original meaning.
2. Preserve speaker intent.
3. Preserve names, numbers, dates and units.
4. Preserve negations.
5. Preserve uncertainty.
6. Preserve questions.
7. Preserve disagreements.
8. Preserve proposals and suggestions.
9. Do NOT turn a proposal into a decision.
10. Do NOT invent task owners.
11. Do NOT invent deadlines.
12. Do NOT add information.
13. Do NOT summarize.
14. Do NOT remove meaningful information.
15. If you are unsure whether a word is wrong, keep the original word.

Return ONLY the refined transcript.

RAW TRANSCRIPT:
{raw_transcript}
"""

    messages = [
        {
            "role": "system",
            "content": "You refine speech-to-text meeting transcripts without changing their meaning."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = refiner_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = refiner_tokenizer(
        text,
        return_tensors="pt"
    ).to(refiner_model.device)

    outputs = refiner_model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False
    )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    refined = refiner_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return refined.strip()

In [ ]:
import re

def split_into_chunks(text, max_words=250):
    """Split a transcript on sentence boundaries into chunks of roughly max_words words,
    so that long meetings are never truncated by the generation limit."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current, count = [], [], 0
    for sentence in sentences:
        n = len(sentence.split())
        if current and count + n > max_words:
            chunks.append(" ".join(current))
            current, count = [], 0
        current.append(sentence)
        count += n
    if current:
        chunks.append(" ".join(current))
    return chunks


def refine_transcript_long(raw_transcript, max_words=250, on_chunk=None):
    """Refine a transcript chunk by chunk (identical to refine_transcript for short inputs)."""
    chunks = split_into_chunks(raw_transcript, max_words)
    refined_parts = []
    for i, chunk in enumerate(chunks, start=1):
        if on_chunk:
            on_chunk(i, len(chunks))
        refined_parts.append(refine_transcript(chunk))
    return " ".join(refined_parts)

In [ ]:
refined_transcript = refine_transcript_long(raw_transcript)

print("===== REFINED TRANSCRIPT =====")
print(refined_transcript)

**PHASE 3. Generating the meeting summary, minutes and the important desicions**

In [ ]:
DOCUMENTATION_MODEL = "Qwen/Qwen2.5-3B-Instruct"

documentation_tokenizer = AutoTokenizer.from_pretrained(DOCUMENTATION_MODEL)

documentation_model = AutoModelForCausalLM.from_pretrained(
    DOCUMENTATION_MODEL,
    device_map={"": DOC_GPU},
    torch_dtype=torch.float16
)

print("Documentation model loaded on cuda:%d" % DOC_GPU)

In [ ]:
DOCUMENTATION_PROMPT = """
You are a HIGH-PRECISION MEETING DOCUMENTATION AND INFORMATION
EXTRACTION SYSTEM.

You are given a REFINED MEETING TRANSCRIPT.

The transcript is the ONLY source of truth.

Your task is to EXTRACT information from the transcript.

You must NEVER invent, assume, infer, predict, or fabricate information.

==================================================
ABSOLUTE SOURCE-OF-TRUTH RULE
==================================================

Every piece of information in your output MUST be directly supported
by the meeting transcript.

If information is not explicitly supported by the transcript:

DO NOT INCLUDE IT.

When there is insufficient evidence, return an empty field/list
instead of guessing.

==================================================
1. SUMMARY
==================================================

Create a concise summary of what was actually discussed.

The summary may contain:
- topics discussed
- problems discussed
- viewpoints expressed
- information shared
- conclusions explicitly reached

The summary MUST NOT:
- invent conclusions
- invent decisions
- invent future plans
- convert suggestions into decisions
- add information not present in the transcript

==================================================
2. MEETING MINUTES
==================================================

Extract the important topics actually discussed during the meeting.

Minutes should describe:
- topics discussed
- important points raised
- relevant arguments
- conclusions explicitly stated

Do NOT add topics that were not discussed.

Do NOT turn every sentence into a minute.

==================================================
3. DECISIONS — EXTREMELY STRICT
==================================================

This is the most important rule.

A DECISION may ONLY be included when the transcript contains
CLEAR AND EXPLICIT evidence that the participants agreed on,
confirmed, approved, selected, finalized, or decided something.

Examples of explicit decision language include:

- "We decided to..."
- "The team agreed to..."
- "We have agreed that..."
- "Let's finalize..."
- "It is confirmed that..."
- "We will proceed with..."
  ONLY when the context clearly establishes this as a confirmed decision.
- "Everyone agreed..."
- "It was finalized that..."
- "The final choice is..."

Do NOT treat the following as decisions:

- "We should..."
- "We could..."
- "Maybe..."
- "Perhaps..."
- "I think..."
- "I suggest..."
- "I recommend..."
- "What if we..."
- "Can we..."
- "Should we..."
- "Let's consider..."
- "It might be better to..."
- "We can possibly..."
- "We may..."
- "I would prefer..."
- "I think we should..."
- "Someone proposed..."
- "X suggested..."
- "X recommended..."

These are proposals, suggestions, opinions, or possibilities,
NOT decisions.

==================================================
DECISION CONSISTENCY TEST
==================================================

Before adding ANY decision, ask:

1. Is this statement explicitly present in the transcript?
2. Is there clear evidence that the participants agreed to it?
3. Was the proposal actually accepted or finalized?
4. Am I interpreting the speaker's intention?
5. Am I converting a suggestion into a decision?

If the answer to question 4 or 5 is YES:

DO NOT INCLUDE IT AS A DECISION.

If there is any uncertainty:

RETURN NO DECISION.

Therefore:

If there are no explicitly confirmed decisions:

"decisions": []

==================================================
4. ACTION ITEMS — EXTREMELY STRICT
==================================================

An ACTION ITEM may ONLY be extracted when the transcript clearly
establishes that some work/task is expected to be performed.

Examples:

"Rahul will prepare the report."

"Priya needs to submit the dataset tomorrow."

"John, please update the presentation."

"The team will send the document to the professor."

These may be action items because a task is explicitly assigned
or clearly committed to.

DO NOT create action items from:

- suggestions
- recommendations
- general discussion
- possibilities
- hypothetical statements
- questions
- ideas that were not accepted
- tasks that were merely mentioned
- tasks discussed as examples

For example:

"We should prepare a report."

DO NOT automatically extract:

"Prepare the report."

Instead, determine whether the transcript contains clear evidence
that the team actually committed to performing that task.

==================================================
5. OWNER
==================================================

Only identify an owner if the transcript explicitly identifies
who is responsible.

Example:

"Rahul will prepare the report."

Owner = Rahul.

Do NOT infer the owner based on:
- who spoke
- who proposed the task
- who normally performs the task
- job title
- department
- context

If the task is explicit but the owner is not specified:

"owner": "Unspecified"

==================================================
6. DEADLINE
==================================================

Only include a deadline if it is explicitly stated.

Example:

"Submit it by Friday."

Deadline = Friday.

Do NOT infer deadlines from:
- meeting date
- urgency
- phrases such as "soon"
- expected workflow
- assumptions

If no deadline is explicitly stated:

"deadline": "Unspecified"

==================================================
7. NO DECISION / NO ACTION MEETING
==================================================

This rule is REQUIRED.

A meeting may contain discussion but NO decisions and NO action items.

For example:

"We discussed whether to use Model A or Model B.
The team has not decided yet."

The correct output is:

"decisions": [],
"action_items": []

DO NOT manufacture a decision simply because the meeting discussed
a possible course of action.

==================================================
8. DISTINGUISH DISCUSSION FROM COMMITMENT
==================================================

Discussion:

"We could deploy this next week."

This does NOT automatically mean:

Decision:
"The team decided to deploy next week."

Likewise:

"We should ask the professor."

does NOT automatically mean:

Action:
"Ask the professor."

Only extract the latter when the transcript clearly establishes
that it was accepted/committed/assigned.

==================================================
9. NO HALLUCINATION
==================================================

NEVER fabricate:

- decisions
- tasks
- names
- owners
- deadlines
- dates
- numbers
- organizations
- conclusions
- commitments
- meeting outcomes

The correct response to missing information is:

EMPTY LIST or "Unspecified"

NOT a guess.

==================================================
10. OUTPUT FORMAT
==================================================

Return ONLY valid JSON.

Do not return:
- Markdown
- explanations
- introductory text
- ```json code fences
- commentary

Use exactly this structure:

{
    "summary": "...",
    "minutes": [
        "..."
    ],
    "decisions": [
        "..."
    ],
    "action_items": [
        {
            "action": "...",
            "owner": "...",
            "deadline": "..."
        }
    ]
}

==================================================
11. FINAL VALIDATION BEFORE OUTPUT
==================================================

Before returning the JSON, perform the following internal checks:

DECISION CHECK:
Every decision must have explicit evidence of agreement,
confirmation, approval, selection, or finalization.

ACTION CHECK:
Every action item must have explicit evidence that the task
is expected/committed/assigned.

OWNER CHECK:
Never infer an owner.

DEADLINE CHECK:
Never infer a deadline.

HALLUCINATION CHECK:
Remove anything that is not directly supported by the transcript.

NO-DECISION CHECK:
If no explicit decision exists, return:

"decisions": []

NO-ACTION CHECK:
If no explicit action item exists, return:

"action_items": []

SOURCE:

{refined_transcript}
"""

In [ ]:
def generate_meeting_record(refined_transcript):

    prompt = DOCUMENTATION_PROMPT + refined_transcript

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise meeting documentation assistant. "
                "Never invent information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Convert messages into the format expected by Qwen
    formatted_prompt = documentation_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text into model inputs
    inputs = documentation_tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    # Move inputs to the model's device
    inputs = {
        key: value.to(documentation_model.device)
        for key, value in inputs.items()
    }

    # Generate response
    with torch.no_grad():

        outputs = documentation_model.generate(
            **inputs,
            max_new_tokens=2000,
            do_sample=False
        )

    # Remove the original prompt from the generated output
    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[-1]:
    ]

    # Convert tokens back to text
    result = documentation_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return result.strip()

In [ ]:
raw_meeting_record = generate_meeting_record(refined_transcript)

print(raw_meeting_record)

In [ ]:
import json
import re
def clean_json_output(text):

    text = text.strip()

    # Remove Markdown code fences if the model added them
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    return text.strip()


def parse_meeting_json(text):
    """Parse the model output as JSON, tolerating code fences and stray text around the object."""
    cleaned = clean_json_output(text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(cleaned[start:end + 1])
        except json.JSONDecodeError:
            pass
    raise ValueError("Model output is not valid JSON")

In [ ]:
meeting_record = parse_meeting_json(raw_meeting_record)
type(meeting_record)

In [ ]:
def _is_none_text(value):
    return isinstance(value, str) and value.strip().lower() in ("", "none", "n/a", "null")


def validate_meeting_record(record):
    """Normalise the model output so every section is a clean list.
    (Fixes the case where the model returns the *string* "None": iterating it would
    otherwise print the letters N, o, n, e one by one.)"""

    if not isinstance(record, dict):
        record = {}

    # Summary
    summary = record.get("summary")
    record["summary"] = "" if summary is None else str(summary)

    # Minutes
    minutes = record.get("minutes")
    if minutes is None or _is_none_text(minutes):
        minutes = []
    elif isinstance(minutes, (str, dict)):
        minutes = [minutes]

    clean_minutes = []
    for minute in minutes:
        if isinstance(minute, str):
            if _is_none_text(minute):
                continue
            minute = {"topic": "Unspecified", "discussion": minute}
        elif not isinstance(minute, dict):
            continue
        clean_minutes.append({
            "topic": minute.get("topic") or "Unspecified",
            "discussion": minute.get("discussion") or ""
        })
    record["minutes"] = clean_minutes

    # Decisions
    decisions = record.get("decisions")
    if decisions is None or _is_none_text(decisions):
        decisions = []
    elif isinstance(decisions, (str, dict)):
        decisions = [decisions]

    clean_decisions = []
    for decision in decisions:
        if isinstance(decision, dict):
            decision = decision.get("decision") or "; ".join(str(v) for v in decision.values())
        if decision is None or _is_none_text(decision):
            continue
        clean_decisions.append(str(decision))
    record["decisions"] = clean_decisions

    # Action items
    actions = record.get("action_items")
    if actions is None or _is_none_text(actions):
        actions = []
    elif isinstance(actions, (str, dict)):
        actions = [actions]

    clean_actions = []
    for action in actions:
        if isinstance(action, str):
            if _is_none_text(action):
                continue
            action = {"task": action}
        elif not isinstance(action, dict):
            continue
        clean_actions.append({
            "task": action.get("task") or "Unspecified",
            "owner": action.get("owner") or "Unspecified",
            "deadline": action.get("deadline") or "Unspecified"
        })
    record["action_items"] = clean_actions

    return record

In [ ]:
meeting_record = validate_meeting_record(
    meeting_record
)

SUMMARY

In [ ]:
print("SUMMARY")
print("=" * 60)

print(meeting_record["summary"])

MINUTES OF THE MEETING

In [ ]:
print("\nMINUTES")
print("=" * 60)

for minute in meeting_record["minutes"]:

    print("\nTopic:", minute["topic"])
    print("Discussion:", minute["discussion"])

KEY DECISIONS

In [ ]:
print("\nKEY DECISIONS")
print("=" * 60)

for decision in meeting_record["decisions"]:

    print("•", decision)

ACTION ITEMS

In [ ]:
print("\nACTION ITEMS")
print("=" * 60)

for i, action in enumerate(
    meeting_record["action_items"],
    start=1
):

    print(f"\n{i}. {action['task']}")
    print("   Owner:", action["owner"])
    print("   Deadline:", action["deadline"])

STRUCTURED OUTPUT (JSON)

In [ ]:
with open(
    os.path.join(WORK_DIR, "meeting_record.json"),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        meeting_record,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved:", os.path.join(WORK_DIR, "meeting_record.json"))

HUMAN READABLE TEXT OUTPUT

In [ ]:
def create_human_readable_record(record):

    text = ""

    text += "MEETING RECORD\n"
    text += "=" * 60 + "\n\n"


    # Summary
    text += "MEETING SUMMARY\n"
    text += "-" * 60 + "\n"
    text += record["summary"] + "\n\n"


    # Minutes
    text += "MEETING MINUTES\n"
    text += "-" * 60 + "\n"

    for minute in record["minutes"]:

        text += f"\nTopic: {minute['topic']}\n"
        text += f"Discussion: {minute['discussion']}\n"


    # Decisions
    text += "\n\nKEY DECISIONS\n"
    text += "-" * 60 + "\n"

    if len(record["decisions"]) == 0:

        text += "No explicit decisions identified.\n"

    else:

        for i, decision in enumerate(
            record["decisions"],
            start=1
        ):

            text += f"{i}. {decision}\n"


    # Action items
    text += "\nACTION ITEMS\n"
    text += "-" * 60 + "\n"

    if len(record["action_items"]) == 0:

        text += "No explicit action items identified.\n"

    else:

        for i, action in enumerate(
            record["action_items"],
            start=1
        ):

            text += f"\n{i}. {action['task']}\n"
            text += f"   Owner: {action['owner']}\n"
            text += f"   Deadline: {action['deadline']}\n"


    return text

In [ ]:
human_readable_record = create_human_readable_record(
    meeting_record
)

print(human_readable_record)

In [ ]:
with open(
    os.path.join(WORK_DIR, "meeting_record.txt"),
    "w",
    encoding="utf-8"
) as f:

    f.write(human_readable_record)

print("Saved:", os.path.join(WORK_DIR, "meeting_record.txt"))

---
# 🌐 PHASE 4 — Web application

Turns the pipeline above into a polished web app (Gradio). It **re-uses the models already loaded in memory**, so there is no second download or reload.

- Upload or record a meeting → a live 4-step progress tracker → results in tabs (Summary, Minutes, Decisions, Action Items, Transcripts)
- Download the results as **JSON**, **TXT** and a styled **HTML report**
- Kaggle will print a public `https://xxxx.gradio.live` link — open it in any browser (valid ~72 h, while this session is alive)

> Keep this notebook session **open** while using the app. "Save & Run All (Commit)" does not keep a web server running — use the interactive session.

In [ ]:
import os, re, gc, json, time, html, shutil, tempfile, subprocess, datetime
import torch
import soundfile as sf
import gradio as gr

print("Gradio", gr.__version__)

_required = [
    "transcriber", "refiner_model", "refiner_tokenizer",
    "documentation_model", "documentation_tokenizer",
    "refine_transcript_long", "generate_meeting_record",
    "parse_meeting_json", "validate_meeting_record", "create_human_readable_record",
]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError("Run the Phase 1-3 cells first (models/functions missing: %s)" % ", ".join(_missing))
print("✅ Models and helper functions found in memory.")

In [ ]:
# ─────────────────────────── Styling ───────────────────────────
# Uses Gradio's CSS variables so it looks right in both light and dark mode.

CARD_CSS = """
.mi-hero{background:linear-gradient(135deg,#4338ca 0%,#7c3aed 55%,#c026d3 115%);color:#fff;border-radius:20px;
  padding:34px 38px;margin-bottom:6px;box-shadow:0 14px 34px -14px rgba(79,70,229,.55)}
.mi-hero h1{margin:0 0 8px;font-size:2.05rem;font-weight:800;letter-spacing:-.02em;color:#fff;line-height:1.15}
.mi-hero p{margin:0;color:rgba(255,255,255,.92);font-size:1.03rem;max-width:760px}
.mi-pills{display:flex;flex-wrap:wrap;gap:8px;margin-top:20px}
.mi-pill{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.3);padding:5px 13px;border-radius:999px;
  font-size:.78rem;font-weight:600;color:#fff;letter-spacing:.01em}

.mi-card{background:var(--background-fill-secondary,#f8fafc);border:1px solid var(--border-color-primary,#e2e8f0);
  border-radius:14px;padding:18px 22px;margin-bottom:12px;color:var(--body-text-color,#0f172a)}
.mi-summary p{margin:0 0 10px;line-height:1.7;font-size:1.02rem}
.mi-summary p:last-child{margin-bottom:0}

.mi-steps{display:flex;align-items:flex-start;justify-content:space-between;gap:6px}
.mi-step{flex:1;display:flex;flex-direction:column;align-items:center;text-align:center;position:relative}
.mi-step:not(:last-child)::after{content:"";position:absolute;top:19px;left:calc(50% + 24px);width:calc(100% - 48px);
  height:3px;border-radius:3px;background:var(--border-color-primary,#e2e8f0)}
.mi-step.done:not(:last-child)::after{background:#6366f1}
.mi-dot{width:38px;height:38px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;
  font-size:.95rem;background:var(--background-fill-primary,#fff);border:2px solid var(--border-color-primary,#cbd5e1);
  color:var(--body-text-color-subdued,#64748b);z-index:1}
.mi-step.done .mi-dot{background:#6366f1;border-color:#6366f1;color:#fff}
.mi-step.active .mi-dot{border-color:#6366f1;color:#6366f1;animation:mi-pulse 1.4s ease-in-out infinite}
.mi-step-name{margin-top:8px;font-size:.8rem;font-weight:600;color:var(--body-text-color-subdued,#64748b)}
.mi-step.active .mi-step-name,.mi-step.done .mi-step-name{color:var(--body-text-color,#0f172a)}
.mi-status-msg{margin-top:16px;padding-top:12px;border-top:1px dashed var(--border-color-primary,#e2e8f0);
  font-size:.9rem;color:var(--body-text-color-subdued,#64748b);text-align:center}
@keyframes mi-pulse{0%,100%{box-shadow:0 0 0 0 rgba(99,102,241,.45)}50%{box-shadow:0 0 0 9px rgba(99,102,241,0)}}

.mi-metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px}
.mi-metric{background:var(--background-fill-secondary,#f8fafc);border:1px solid var(--border-color-primary,#e2e8f0);
  border-radius:12px;padding:12px 14px}
.mi-metric-value{font-size:1.4rem;font-weight:800;color:var(--body-text-color,#0f172a);letter-spacing:-.01em}
.mi-metric-label{font-size:.72rem;text-transform:uppercase;letter-spacing:.06em;font-weight:600;
  color:var(--body-text-color-subdued,#64748b);margin-top:2px}

.mi-minute{display:flex;gap:14px;padding:16px 18px;margin-bottom:10px;border-radius:14px;
  background:var(--background-fill-secondary,#f8fafc);border:1px solid var(--border-color-primary,#e2e8f0);
  border-left:4px solid #6366f1;color:var(--body-text-color,#0f172a)}
.mi-minute-num{flex:none;width:28px;height:28px;border-radius:8px;background:rgba(99,102,241,.14);color:#6366f1;
  font-weight:800;font-size:.85rem;display:flex;align-items:center;justify-content:center}
.mi-minute-topic{font-weight:700;font-size:1.02rem;margin-bottom:4px}
.mi-minute-text{line-height:1.65;color:var(--body-text-color,#0f172a);opacity:.9}

.mi-decision{display:flex;gap:12px;align-items:flex-start;padding:14px 18px;margin-bottom:10px;border-radius:14px;
  background:var(--background-fill-secondary,#f8fafc);border:1px solid var(--border-color-primary,#e2e8f0);
  color:var(--body-text-color,#0f172a);line-height:1.6}
.mi-check{flex:none;width:24px;height:24px;border-radius:50%;background:#10b981;color:#fff;font-size:.8rem;font-weight:800;
  display:flex;align-items:center;justify-content:center;margin-top:1px}

.mi-table-wrap{overflow-x:auto;border-radius:14px;border:1px solid var(--border-color-primary,#e2e8f0)}
.mi-table-wrap table,.mi-table-wrap th,.mi-table-wrap td{border-left:0 !important;border-right:0 !important;border-top:0 !important}
.mi-table-wrap table{border:0 !important;margin:0 !important}
.mi-table-wrap tr,.mi-table-wrap thead,.mi-table-wrap tbody{border:0 !important;background:transparent !important}
.mi-table{width:100%;border-collapse:collapse;font-size:.95rem;color:var(--body-text-color,#0f172a)}
.mi-table th{text-align:left;padding:12px 16px;font-size:.72rem;text-transform:uppercase;letter-spacing:.06em;
  background:var(--background-fill-secondary,#f1f5f9);color:var(--body-text-color-subdued,#64748b);
  border-bottom:1px solid var(--border-color-primary,#e2e8f0) !important}
.mi-table td{padding:14px 16px;border-bottom:1px solid var(--border-color-primary,#e2e8f0) !important;vertical-align:top;line-height:1.55}
.mi-table tr:last-child td{border-bottom:none !important}
.mi-badge{display:inline-block;padding:3px 11px;border-radius:999px;font-size:.8rem;font-weight:600;
  background:rgba(99,102,241,.14);color:#6366f1}
.mi-badge.muted{background:rgba(148,163,184,.18);color:var(--body-text-color-subdued,#64748b);font-weight:500}

.mi-empty{text-align:center;padding:38px 20px;border:1.5px dashed var(--border-color-primary,#cbd5e1);border-radius:14px;
  color:var(--body-text-color-subdued,#64748b);font-size:.95rem}
.mi-empty-icon{font-size:1.9rem;margin-bottom:6px}
.mi-tip{font-size:.85rem;color:var(--body-text-color-subdued,#64748b);line-height:1.6;padding:4px 4px 0}
.mi-footer{text-align:center;font-size:.8rem;color:var(--body-text-color-subdued,#64748b);padding:18px 0 4px}
"""

APP_CSS = CARD_CSS + """
.gradio-container{max-width:1200px !important;margin:0 auto !important}
footer{display:none !important}
"""

REPORT_CSS = """
:root{--background-fill-primary:#ffffff;--background-fill-secondary:#f8fafc;--border-color-primary:#e2e8f0;
  --body-text-color:#0f172a;--body-text-color-subdued:#64748b}
*{box-sizing:border-box}
body{font-family:Inter,'Segoe UI',system-ui,-apple-system,sans-serif;background:#eef2ff;margin:0;padding:32px 16px;color:#0f172a}
.mi-page{max-width:920px;margin:0 auto}
h2.mi-h{font-size:1.15rem;margin:30px 0 12px;font-weight:800;letter-spacing:-.01em}
.mi-pre{white-space:pre-wrap;word-wrap:break-word;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.85rem;
  line-height:1.6;background:#fff;border:1px solid #e2e8f0;border-radius:12px;padding:16px;margin:10px 0 0}
details.mi-details{margin-top:12px}
details.mi-details summary{cursor:pointer;font-weight:700;padding:10px 0}
""" + CARD_CSS

In [ ]:
# ─────────────────────────── HTML renderers ───────────────────────────

def esc(x):
    return html.escape("" if x is None else str(x))


def fmt_duration(seconds):
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


STAGES = [
    ("Prepare audio", "🎧"),
    ("Transcribe",    "📝"),
    ("Refine",        "✨"),
    ("Document",      "📋"),
]


def render_status(stage, message=""):
    """stage: -1 = idle, 0..3 = that step is running, 4 = everything finished."""
    steps = []
    for i, (name, icon) in enumerate(STAGES):
        if stage >= 4 or i < stage:
            cls, mark = "done", "✓"
        elif i == stage:
            cls, mark = "active", icon
        else:
            cls, mark = "todo", str(i + 1)
        steps.append(
            f'<div class="mi-step {cls}"><div class="mi-dot">{mark}</div>'
            f'<div class="mi-step-name">{name}</div></div>'
        )
    return (
        '<div class="mi-card"><div class="mi-steps">' + "".join(steps) + "</div>"
        f'<div class="mi-status-msg">{esc(message)}</div></div>'
    )


def render_metrics(duration=None, words=None, decisions=None, actions=None):
    def tile(label, value):
        return (f'<div class="mi-metric"><div class="mi-metric-value">{esc(value)}</div>'
                f'<div class="mi-metric-label">{esc(label)}</div></div>')
    dash = "—"
    return '<div class="mi-metrics">' + "".join([
        tile("Audio length",  fmt_duration(duration) if duration is not None else dash),
        tile("Words",         f"{words:,}" if words is not None else dash),
        tile("Decisions",     decisions if decisions is not None else dash),
        tile("Action items",  actions if actions is not None else dash),
    ]) + "</div>"


def render_empty(message, icon="🗂️"):
    return f'<div class="mi-empty"><div class="mi-empty-icon">{icon}</div><div>{esc(message)}</div></div>'


def render_summary(record):
    text = (record.get("summary") or "").strip()
    if not text:
        return render_empty("No summary was generated.", "📝")
    paras = "".join(f"<p>{esc(p)}</p>" for p in text.split("\n") if p.strip())
    return f'<div class="mi-card mi-summary">{paras}</div>'


def render_minutes(record):
    minutes = record.get("minutes") or []
    if not minutes:
        return render_empty("No distinct discussion topics were identified.", "🗒️")
    out = []
    for i, m in enumerate(minutes, start=1):
        out.append(
            f'<div class="mi-minute"><div class="mi-minute-num">{i}</div><div>'
            f'<div class="mi-minute-topic">{esc(m.get("topic"))}</div>'
            f'<div class="mi-minute-text">{esc(m.get("discussion"))}</div></div></div>'
        )
    return "".join(out)


def render_decisions(record):
    decisions = record.get("decisions") or []
    if not decisions:
        return render_empty("No explicit decisions were identified in this meeting.", "⚖️")
    return "".join(
        f'<div class="mi-decision"><div class="mi-check">✓</div><div>{esc(d)}</div></div>'
        for d in decisions
    )


def _badge(value):
    value = str(value or "Unspecified")
    cls = "mi-badge muted" if value.strip().lower() == "unspecified" else "mi-badge"
    return f'<span class="{cls}">{esc(value)}</span>'


def render_actions(record):
    actions = record.get("action_items") or []
    if not actions:
        return render_empty("No explicit action items were identified in this meeting.", "✅")
    rows = "".join(
        f"<tr><td>{i}</td><td>{esc(a.get('task'))}</td><td>{_badge(a.get('owner'))}</td><td>{_badge(a.get('deadline'))}</td></tr>"
        for i, a in enumerate(actions, start=1)
    )
    return (
        '<div class="mi-table-wrap"><table class="mi-table"><thead><tr>'
        "<th>#</th><th>Task</th><th>Owner</th><th>Deadline</th></tr></thead>"
        f"<tbody>{rows}</tbody></table></div>"
    )


def build_html_report(record, raw, refined, meta):
    """Standalone, shareable HTML report (opens in any browser, prints cleanly to PDF)."""
    stamp = datetime.datetime.now().strftime("%d %b %Y, %H:%M")
    return (
        '<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        "<title>Meeting Record</title><style>" + REPORT_CSS + "</style></head><body><div class='mi-page'>"
        '<div class="mi-hero"><h1>Meeting Record</h1>'
        f"<p>Generated on {esc(stamp)} · Audio length {esc(meta.get('duration', '—'))} · {esc(meta.get('words', '—'))} words</p></div>"
        '<h2 class="mi-h">Summary</h2>' + render_summary(record) +
        '<h2 class="mi-h">Meeting minutes</h2>' + render_minutes(record) +
        '<h2 class="mi-h">Key decisions</h2>' + render_decisions(record) +
        '<h2 class="mi-h">Action items</h2>' + render_actions(record) +
        '<h2 class="mi-h">Transcripts</h2>'
        '<details class="mi-details"><summary>Refined transcript</summary>'
        f'<div class="mi-pre">{esc(refined)}</div></details>'
        '<details class="mi-details"><summary>Raw transcript</summary>'
        f'<div class="mi-pre">{esc(raw)}</div></details>'
        "</div></body></html>"
    )

In [ ]:
# ─────────────────────────── Backend pipeline ───────────────────────────

EMPTY_TEXT = {
    "summary":   "Your meeting summary will appear here.",
    "minutes":   "Organised minutes will appear here.",
    "decisions": "Explicit decisions will appear here.",
    "actions":   "Action items with owners and deadlines will appear here.",
}


def blank_state():
    return {
        "status":    render_status(-1, "Waiting for a recording…"),
        "metrics":   render_metrics(),
        "raw":       "",
        "refined":   "",
        "summary":   render_empty(EMPTY_TEXT["summary"], "📝"),
        "minutes":   render_empty(EMPTY_TEXT["minutes"], "🗒️"),
        "decisions": render_empty(EMPTY_TEXT["decisions"], "⚖️"),
        "actions":   render_empty(EMPTY_TEXT["actions"], "✅"),
        "json": None, "txt": None, "html": None,
    }


def pack(s):
    return (s["status"], s["metrics"], s["raw"], s["refined"], s["summary"],
            s["minutes"], s["decisions"], s["actions"], s["json"], s["txt"], s["html"])


def convert_to_wav(src_path, work_dir):
    dst = os.path.join(work_dir, "input_16k_mono.wav")
    proc = subprocess.run(
        ["ffmpeg", "-y", "-i", src_path, "-ar", "16000", "-ac", "1", dst],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
    )
    if proc.returncode != 0 or not os.path.exists(dst):
        raise gr.Error("Could not read this audio file. Please try a different format (mp3, wav, m4a).")
    return dst


def run_pipeline(audio_path):
    """Generator: yields UI updates after every stage so the page shows live progress."""
    if not audio_path:
        raise gr.Error("Please upload or record a meeting first.")

    s = blank_state()
    t0 = time.time()
    work_dir = tempfile.mkdtemp(prefix="meeting_")

    try:
        # 1 ── prepare audio
        s["status"] = render_status(0, "Converting audio to 16 kHz mono WAV…")
        yield pack(s)
        wav = convert_to_wav(audio_path, work_dir)
        info = sf.info(wav)
        duration = info.frames / info.samplerate
        s["metrics"] = render_metrics(duration=duration)

        # 2 ── transcribe
        s["status"] = render_status(1, "Transcribing speech with Whisper…")
        yield pack(s)
        raw = transcriber(wav, return_timestamps=True, chunk_length_s=30, batch_size=8)["text"].strip()
        if not raw:
            raise gr.Error("No speech was detected in this recording.")
        words = len(raw.split())
        s["raw"] = raw
        s["metrics"] = render_metrics(duration=duration, words=words)

        # 3 ── refine (chunked so long meetings are not truncated)
        s["status"] = render_status(2, "Correcting technical terms and recognition errors…")
        yield pack(s)
        refined = refine_transcript_long(raw)
        s["refined"] = refined

        # 4 ── structured documentation
        s["status"] = render_status(3, "Extracting summary, minutes, decisions and action items…")
        yield pack(s)
        raw_record = generate_meeting_record(refined)
        try:
            record = validate_meeting_record(parse_meeting_json(raw_record))
        except ValueError:
            record = validate_meeting_record({
                "summary": "⚠️ The model's answer could not be parsed as structured data. Raw model output:\n" + raw_record
            })

        # files
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        json_path = os.path.join(work_dir, f"meeting_record_{stamp}.json")
        txt_path  = os.path.join(work_dir, f"meeting_record_{stamp}.txt")
        html_path = os.path.join(work_dir, f"meeting_report_{stamp}.html")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=4, ensure_ascii=False)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(create_human_readable_record(record))
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(build_html_report(record, raw, refined,
                                      {"duration": fmt_duration(duration), "words": f"{words:,}"}))

        # done
        s.update({
            "status":    render_status(4, f"Finished in {fmt_duration(time.time() - t0)}"),
            "metrics":   render_metrics(duration, words, len(record["decisions"]),
                                        len(record["action_items"])),
            "summary":   render_summary(record),
            "minutes":   render_minutes(record),
            "decisions": render_decisions(record),
            "actions":   render_actions(record),
            "json": json_path, "txt": txt_path, "html": html_path,
        })
        yield pack(s)

    except gr.Error:
        raise
    except torch.cuda.OutOfMemoryError:
        raise gr.Error("GPU ran out of memory. Try a shorter recording, or switch the session to GPU T4 x2.")
    except Exception as e:
        raise gr.Error(f"Processing failed: {type(e).__name__}: {e}")
    finally:
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────── User interface ───────────────────────────

HERO_HTML = """
<div class="mi-hero">
  <h1>🧠 Meeting Intelligence System</h1>
  <p>Turn any meeting recording into an accurate transcript, concise summary, organised minutes,
     confirmed decisions and owner-assigned action items — automatically.</p>
  <div class="mi-pills">
    <span class="mi-pill">Whisper speech-to-text</span>
    <span class="mi-pill">Domain-aware transcript refinement</span>
    <span class="mi-pill">Hallucination-guarded extraction</span>
    <span class="mi-pill">JSON · TXT · HTML export</span>
  </div>
</div>
"""

TIPS_HTML = """
<div class="mi-tip">
  <b>Tips</b><br>
  • Clear audio gives the best accuracy; MP3, WAV, M4A, FLAC and OGG are supported.<br>
  • Only <i>explicitly agreed</i> decisions and <i>clearly assigned</i> tasks are reported — suggestions are never promoted to decisions.<br>
  • Missing owners or deadlines are shown as <i>Unspecified</i> rather than guessed.
</div>
"""

FOOTER_HTML = """
<div class="mi-footer">
  Whisper&nbsp;small · Qwen2.5-7B-Instruct (4-bit) · Qwen2.5-3B-Instruct · Built with Gradio<br>
  Audio is processed inside this notebook session and is not sent to any third-party AI service.
</div>
"""

theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="violet",
    neutral_hue="slate",
    radius_size="lg",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
)

_blank = blank_state()

with gr.Blocks(theme=theme, css=APP_CSS, title="Meeting Intelligence System") as demo:

    gr.HTML(HERO_HTML)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Meeting recording",
            )
            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)
                run_btn   = gr.Button("🚀  Process meeting", variant="primary", scale=3, size="lg")
            gr.HTML(TIPS_HTML)

        with gr.Column(scale=6):
            status_html  = gr.HTML(_blank["status"])
            metrics_html = gr.HTML(_blank["metrics"])

    with gr.Tabs():
        with gr.Tab("📝 Summary"):
            summary_html = gr.HTML(_blank["summary"])
        with gr.Tab("🗒️ Minutes"):
            minutes_html = gr.HTML(_blank["minutes"])
        with gr.Tab("⚖️ Decisions"):
            decisions_html = gr.HTML(_blank["decisions"])
        with gr.Tab("✅ Action items"):
            actions_html = gr.HTML(_blank["actions"])
        with gr.Tab("🎙️ Transcripts"):
            with gr.Row():
                raw_output = gr.Textbox(label="Raw transcript (Whisper)", lines=16, show_copy_button=True)
                refined_output = gr.Textbox(label="Refined transcript (AI-corrected)", lines=16, show_copy_button=True)

    gr.Markdown("### 📥 Download")
    with gr.Row():
        html_file = gr.File(label="Report (HTML — printable to PDF)")
        json_file = gr.File(label="Structured data (JSON)")
        txt_file  = gr.File(label="Plain-text minutes (TXT)")

    gr.HTML(FOOTER_HTML)

    outputs = [status_html, metrics_html, raw_output, refined_output,
               summary_html, minutes_html, decisions_html, actions_html,
               json_file, txt_file, html_file]

    run_btn.click(
        run_pipeline, inputs=audio_input, outputs=outputs,
        show_progress="hidden", concurrency_limit=1,
    )
    clear_btn.click(
        lambda: (None,) + pack(blank_state()),
        inputs=None, outputs=[audio_input] + outputs,
    )

print("UI built.")

### 🚀 Launch
Run the next cell and open the printed **public URL** (`https://….gradio.live`). The cell keeps running while the app is live — stop it with the ■ button when you are done.

In [ ]:
demo.queue(max_size=8).launch(
    share=True,     # public link; requires Internet = On in Kaggle
    debug=True      # keeps the cell alive and prints errors
)